# AED Full GPU Benchmark
## Attention Entropy Divergence — Hallucination Detection

**Runtime:** ~15 min on T4 GPU (free Colab tier)  
**Output:** `results/benchmark_results.json` — download and commit to repo

### What this does
1. Loads **Pythia-160m** (the model used in the paper)
2. Loads **HaluEval QA** (the benchmark used in the paper)
3. Extracts per-head Shannon entropy and cross-layer KL divergence for every sample
4. Trains BiLSTM and logistic regression classifiers on the per-layer feature sequences
5. Reports AUROC, FPR@90%TPR, F1, and latency

### Before you run
**Runtime → Change runtime type → T4 GPU**  
This is required. CPU inference on Pythia-160m + 500 HaluEval samples would take ~2 hours.

In [ ]:
import subprocess, sys

# Verify GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('   This notebook will time out on CPU.')
else:
    print('✓ GPU detected:', result.stdout.strip())

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
%pip install -q transformers datasets accelerate
print('✓ Dependencies installed')

In [ ]:
import os, sys

REPO = 'Language-Model-Hallucination-Detection-via-Entropy-Divergence'
if not os.path.exists(REPO):
    !git clone https://github.com/A-Kuo/{REPO}.git
else:
    !git -C {REPO} pull

os.chdir(REPO)
sys.path.insert(0, 'v1')
print('✓ Repo ready. Working directory:', os.getcwd())

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── Run the full benchmark ─────────────────────────────────────────────────
# Model: EleutherAI/pythia-160m (12 layers, 12 heads, 160M params)
# Dataset: HaluEval QA split (500 samples = 250 correct + 250 hallucinated)
# Expected time: ~12-18 minutes on T4
# ──────────────────────────────────────────────────────────────────────────
!python v1/run_experiment.py \
    --mode full \
    --model EleutherAI/pythia-160m \
    --max-samples 500 \
    --output results/benchmark_results.json

In [ ]:
import json

with open('results/benchmark_results.json') as f:
    r = json.load(f)

print('='*60)
print('  FINAL RESULTS')
print('='*60)
print(f'  Model:             {r["model_name"]}')
print(f'  Device:            {r["device"]}')
print(f'  Samples:           {r["num_samples"]}')
print(f'  Layers × Heads:    {r["num_layers"]} × {r["num_heads"]}')
print()
print(f'  AED (BiLSTM):')
print(f'    AUROC            = {r["aed_auroc"]:.4f}')
print(f'    FPR@90%TPR       = {r["aed_fpr90"]:.4f}')
print(f'    Latency          = {r["aed_latency_ms"]:.1f} ms/sample')
print()
print(f'  Baseline (LogReg on global stats):')
print(f'    AUROC            = {r["logreg_auroc"]:.4f}')
print(f'    FPR@90%TPR       = {r["logreg_fpr90"]:.4f}')
print('='*60)
print()
print('Paper placeholders to fill in paper.tex:')
print(f'  AED BiLSTM AUROC:      \\RESULT{{{r["aed_auroc"]:.3f}}}')
print(f'  LogReg AUROC:          \\RESULT{{{r["logreg_auroc"]:.3f}}}')
print(f'  Latency (ms/sample):   {r["aed_latency_ms"]:.1f}')

In [ ]:
# ── Visualize per-layer entropy patterns ──────────────────────────────────
# This produces the figure for the paper: entropy by layer for hallucinated
# vs. non-hallucinated samples. Expected: hallucinated samples show higher
# entropy in middle-to-late layers (6-9 of 12).

import numpy as np
import matplotlib.pyplot as plt
from run_experiment import extract_features, QUICK_PAIRS
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print('Loading model for visualization...')
tokenizer = AutoTokenizer.from_pretrained('EleutherAI/pythia-160m')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained('EleutherAI/pythia-160m', output_attentions=True)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Use a small representative subset for visualization
# Load from the full HaluEval if available, else use synthetic pairs
try:
    from datasets import load_dataset
    ds = load_dataset('pminervini/HaluEval', 'qa_samples', split='data')
    viz_pairs = []
    for row in ds:
        if len(viz_pairs) >= 40: break
        if row.get('question') and row.get('right_answer'):
            viz_pairs.append((row['question'], row['right_answer'], 0))
        if row.get('question') and row.get('hallucinated_answer') and len(viz_pairs) < 40:
            viz_pairs.append((row['question'], row['hallucinated_answer'], 1))
    print(f'Loaded {len(viz_pairs)} HaluEval samples for visualization')
except Exception:
    viz_pairs = QUICK_PAIRS
    print('Using synthetic pairs for visualization')

num_layers = model.config.num_hidden_layers
correct_entropy = np.zeros(num_layers)
halluc_entropy = np.zeros(num_layers)
correct_count = halluc_count = 0

for ctx, cont, label in viz_pairs:
    feats = extract_features(model, tokenizer, ctx, cont, device)
    layer_entropy = feats[:num_layers]  # first num_layers values are entropy
    if label == 0:
        correct_entropy += layer_entropy
        correct_count += 1
    else:
        halluc_entropy += layer_entropy
        halluc_count += 1

correct_entropy /= max(correct_count, 1)
halluc_entropy /= max(halluc_count, 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

layers = range(1, num_layers + 1)
ax1.plot(layers, correct_entropy, 'b-o', markersize=4, label='Non-hallucinated', linewidth=2)
ax1.plot(layers, halluc_entropy, 'r-o', markersize=4, label='Hallucinated', linewidth=2)
ax1.set_xlabel('Transformer Layer')
ax1.set_ylabel('Mean Attention Entropy (bits)')
ax1.set_title('Per-Layer Attention Entropy by Hallucination Label')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Entropy difference
diff = halluc_entropy - correct_entropy
colors = ['red' if d > 0 else 'blue' for d in diff]
ax2.bar(layers, diff, color=colors, alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Transformer Layer')
ax2.set_ylabel('Entropy Difference (hallucinated − correct)')
ax2.set_title('AED Signal by Layer: Positive = Hallucination Indicator')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/layer_entropy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: results/layer_entropy_analysis.png')
print('  → This figure goes in the paper as Figure 1 or Figure 2')

In [ ]:
# ── Ablation: entropy-only vs KL-only vs both ─────────────────────────────
# This fills in the ablation table in the paper.

import json
import numpy as np
from run_experiment import (
    load_halueval_qa, extract_features, BiLSTMClassifier, auroc, fpr_at_tpr
)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Model should already be loaded from above cell
print('Running ablation study...')
pairs = load_halueval_qa(max_samples=200)  # smaller for speed

# Extract full features
features, labels = [], []
for ctx, cont, label in pairs:
    feat = extract_features(model, tokenizer, ctx, cont, device)
    features.append(feat)
    labels.append(label)

X = np.array(features)   # (N, num_layers * 2)
y = np.array(labels)
num_layers = X.shape[1] // 2

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

results_ablation = {}
for name, cols in [
    ('Entropy only',    list(range(num_layers))),
    ('KL only',         list(range(num_layers, num_layers * 2))),
    ('Both (full AED)', list(range(num_layers * 2))),
]:
    Xtr = X_train[:, cols]
    Xte = X_test[:, cols]
    clf = BiLSTMClassifier(input_dim=len(cols))
    clf.fit(Xtr, y_train.astype(np.float32))
    proba = clf.predict_proba(Xte)
    auc = auroc(y_test, proba)
    fpr = fpr_at_tpr(y_test, proba)
    results_ablation[name] = {'auroc': round(auc, 4), 'fpr90': round(fpr, 4)}
    print(f'  {name:<25}  AUROC={auc:.4f}  FPR@90%TPR={fpr:.4f}')

print()
print('Ablation table for paper.tex:')
print('\\midrule')
for name, res in results_ablation.items():
    baseline = results_ablation['Both (full AED)']['auroc']
    delta = res['auroc'] - baseline
    print(f'{name:<30}  & {res["auroc"]:.3f} & {delta:+.3f} \\\\')

with open('results/ablation_results.json', 'w') as f:
    json.dump(results_ablation, f, indent=2)
print('\n✓ Saved: results/ablation_results.json')

In [ ]:
# ── Download results ───────────────────────────────────────────────────────
# Download the JSON files to your local machine, then:
#   1. Commit them to the repo under results/
#   2. Fill in paper.tex \RESULT{} placeholders with the real numbers
#   3. Fill in the ablation table
#   4. Use layer_entropy_analysis.png as Figure 1

from google.colab import files
import os

for fname in ['results/benchmark_results.json',
              'results/ablation_results.json',
              'results/layer_entropy_analysis.png']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'✓ Downloading: {fname}')
    else:
        print(f'  (not found, skipping): {fname}')

print()
print('Next steps:')
print('  1. git add results/benchmark_results.json results/ablation_results.json')
print('  2. git commit -m "results: GPU benchmark on Pythia-160m + HaluEval"')
print('  3. Fill paper/paper.tex \\RESULT{} placeholders with these numbers')
print('  4. Fill ablation table in paper.tex')
print('  5. pdflatex paper/paper.tex → review → upload to arxiv.org/submit')